# VibeCheck — CLIP-LoRA aesthetic classifier

This notebook trains a small **LoRA-fine-tuned CLIP-ViT-B/32** classifier
for 8 aesthetic categories, and compares it against two baselines:

1. **Groq / Llama-4 Scout** — current production multimodal LLM (3rd-party).
2. **CLIP zero-shot** — frozen CLIP with hand-written class prompts.
3. **CLIP linear probe** — frozen CLIP backbone + trained logistic-regression head.
4. **CLIP + LoRA (ours)** — rank-8 adapters fine-tuned on the vision tower.

Run on **Colab T4** (or local Mac MPS). The full pipeline takes ~5–10 min
on a T4 with ~80 photos.

## Pipeline

```
data/eval/<aesthetic>/*.jpg  →  CLIP image features  ─┬─►  zero-shot (prompts)
                                                       ├─►  linear probe (sklearn)
                                                       └─►  LoRA fine-tune (peft)
                                                              │
                                                              ▼
                                                     compare on held-out test set
                                                     vs. Groq Vision baseline
```

**Open in Colab:** upload this notebook, then upload the `data/eval/` folder
to Colab as well (or mount Google Drive).

## 1. Setup

Install dependencies. On Colab the first run pulls down ~2 GB of weights;
subsequent runs are cached.

In [ ]:
# Colab-only: uncomment the next line on Colab. Locally, install in your venv.
# !pip install -q torch torchvision open_clip_torch transformers peft scikit-learn matplotlib seaborn pillow-heif tqdm

import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    top_k_accuracy_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# HEIC support (iPhone photos)
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    print('pillow-heif not installed; HEIC files will be skipped')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {DEVICE}')

## 2. Data loading

Walks `data/eval/<aesthetic>/*.{jpg,png,heic}` and builds a flat list of
`(image_path, label)` pairs.

**Important**: this notebook expects the `data/eval/` directory to be
alongside the notebook (or one level up). On Colab, upload the directory
or mount Drive.

In [ ]:
# Resolve data root: works in repo (notebooks/ next to data/) and on Colab
# (where you upload data/eval/ to the working directory).
CANDIDATES = [Path('data/eval'), Path('../data/eval'), Path('/content/data/eval')]
DATA_ROOT = next((p for p in CANDIDATES if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find data/eval/. Upload it to Colab or run from repo root.')
print(f'Data root: {DATA_ROOT.resolve()}')

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp'}

# Class list must match data/eval/<folder> names. Order = label index.
CLASSES = sorted(
    d.name for d in DATA_ROOT.iterdir()
    if d.is_dir() and not d.name.startswith('.')
)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f'Found {len(CLASSES)} classes: {CLASSES}')

items: list[tuple[Path, int]] = []
for cls in CLASSES:
    cls_dir = DATA_ROOT / cls
    for p in cls_dir.iterdir():
        if p.suffix.lower() in IMG_EXTS:
            items.append((p, CLASS_TO_IDX[cls]))

print(f'Total photos: {len(items)}')
print('Per-class counts:')
for cls, count in Counter(CLASSES[lbl] for _, lbl in items).items():
    print(f'  {cls:30s} {count}')

### 2.1 Train / val / test split

Stratified split so each class appears proportionally in every split. With
~8 photos per class we use a 60/20/20 split (≈5/1/1–2 per class). Tight
but workable; we'll also do k-fold CV later for the linear probe.

In [ ]:
paths = np.array([str(p) for p, _ in items])
labels = np.array([lbl for _, lbl in items])

# 60/20/20 stratified split
X_temp, X_test, y_temp, y_test = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED
)  # 0.25 of 0.8 = 0.2 overall

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## 3. Load CLIP

We use the HuggingFace `transformers` CLIP (`openai/clip-vit-base-patch32`)
because it integrates cleanly with `peft` for LoRA. Same weights as OpenAI's
released ViT-B/32 model.

Model size: ~88M params total, ~88M frozen. With LoRA rank=8 we'll add
~0.3M trainable params (~0.3% of the model).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

n_params = sum(p.numel() for p in clip_model.parameters())
print(f'CLIP loaded: {n_params/1e6:.1f}M params total')

### 3.1 Image feature extraction (cached)

Run CLIP on every photo once and cache the image embeddings. Then the
linear probe and the zero-shot evaluation reuse these — no re-encoding.

In [ ]:
def load_image(path: str) -> Image.Image:
    return Image.open(path).convert('RGB')

@torch.no_grad()
def encode_images(paths: list[str], batch_size: int = 16) -> np.ndarray:
    feats = []
    for i in tqdm(range(0, len(paths), batch_size), desc='encoding'):
        batch = [load_image(p) for p in paths[i:i+batch_size]]
        inputs = clip_processor(images=batch, return_tensors='pt').to(DEVICE)
        f = clip_model.get_image_features(**inputs)
        f = F.normalize(f, dim=-1)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

feats_train = encode_images(X_train.tolist())
feats_val = encode_images(X_val.tolist())
feats_test = encode_images(X_test.tolist())
print(f'Feature dim: {feats_train.shape[1]}')

## 4. Baseline 1 — CLIP zero-shot

Hand-write a prompt per class. Encode prompts with CLIP's text tower,
compute cosine similarity to image features, take argmax.

This is the **untrained** baseline. If LoRA can't beat this, fine-tuning
isn't earning its keep.

In [ ]:
# Class prompts. Multiple templates per class, averaged, is a common
# zero-shot strength trick from the original CLIP paper.
PROMPT_TEMPLATES = [
    'a photo of a {} aesthetic room',
    'a photo in the {} aesthetic',
    'a {} style space',
    'an outfit in the {} aesthetic',
    'a {} vibe',
]

# Human-readable class names for prompt insertion
CLASS_PROMPT_NAMES = {
    'minimalist': 'minimalist',
    'dark_academia': 'dark academia',
    'cottagecore': 'cottagecore',
    'grunge': 'grunge',
    'coastal_grandmother': 'coastal grandmother',
    'scandinavian': 'scandinavian',
    'mid_century_modern': 'mid-century modern',
    'japandi': 'japandi',
}

@torch.no_grad()
def class_prompt_features() -> np.ndarray:
    """Average the encoded prompt templates per class, l2-normalize."""
    per_class = []
    for cls in CLASSES:
        name = CLASS_PROMPT_NAMES[cls]
        prompts = [t.format(name) for t in PROMPT_TEMPLATES]
        inputs = clip_processor(text=prompts, return_tensors='pt', padding=True).to(DEVICE)
        f = clip_model.get_text_features(**inputs)
        f = F.normalize(f, dim=-1).mean(dim=0)
        f = F.normalize(f, dim=-1)
        per_class.append(f.cpu().numpy())
    return np.stack(per_class, axis=0)

text_proto = class_prompt_features()  # (K, D)

def zero_shot_predict(feats: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (predicted_labels, full_logits) using cosine similarity."""
    sims = feats @ text_proto.T  # (N, K)
    return sims.argmax(axis=1), sims

zs_pred, zs_logits = zero_shot_predict(feats_test)
zs_acc = accuracy_score(y_test, zs_pred)
zs_top3 = top_k_accuracy_score(y_test, zs_logits, k=3, labels=list(range(len(CLASSES))))
print(f'CLIP zero-shot — top-1: {zs_acc:.3f}  top-3: {zs_top3:.3f}')

## 5. Baseline 2 — CLIP linear probe

Train a multinomial logistic regression on the cached features. Backbone
frozen. This is the standard "how good are CLIP features for *this* task"
measurement.

With ~50 training photos and 512-D features, regularization matters a lot.
We sweep `C` (inverse of L2 strength) and pick by val accuracy.

In [ ]:
best_lp = None
for C in [0.01, 0.1, 1.0, 10.0]:
    lp = LogisticRegression(
        C=C, max_iter=2000, multi_class='multinomial', random_state=SEED
    )
    lp.fit(feats_train, y_train)
    val_acc = lp.score(feats_val, y_val)
    print(f'  C={C:>6}  val_acc={val_acc:.3f}')
    if best_lp is None or val_acc > best_lp[1]:
        best_lp = (lp, val_acc, C)

lp_model, _, lp_C = best_lp
print(f'\nBest C={lp_C}')
lp_pred = lp_model.predict(feats_test)
lp_logits = lp_model.predict_proba(feats_test)
lp_acc = accuracy_score(y_test, lp_pred)
lp_top3 = top_k_accuracy_score(y_test, lp_logits, k=3, labels=list(range(len(CLASSES))))
print(f'CLIP linear probe — top-1: {lp_acc:.3f}  top-3: {lp_top3:.3f}')

## 6. Ours — CLIP + LoRA fine-tune

Wrap the CLIP vision encoder with LoRA adapters via `peft`. Train a small
classification head end-to-end with the LoRA params; everything else stays
frozen.

**Hyperparameters** (defaults — feel free to sweep):
- LoRA rank `r=8`, alpha=16, dropout=0.1
- Target modules: `q_proj, k_proj, v_proj, out_proj` in the vision transformer
- Optimizer: AdamW, lr=1e-4 (LoRA params), lr=1e-3 (classifier head)
- Loss: cross-entropy with optional label smoothing
- Epochs: 30 with early stopping on val accuracy
- Batch size: 16 (use 32+ if you have more data)
- Augmentation: light — random resized crop + horizontal flip

In [ ]:
from peft import LoraConfig, get_peft_model
from torchvision import transforms

# Image transforms: CLIP's own processor handles normalization, but for
# end-to-end training we want augmentation on the train split.
MEAN = clip_processor.image_processor.image_mean
STD = clip_processor.image_processor.image_std
IMG_SIZE = clip_processor.image_processor.crop_size['height']

train_tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class VibeDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths = paths
        self.labels = labels
        self.tfm = tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = load_image(self.paths[i])
        return self.tfm(img), int(self.labels[i])

train_ds = VibeDataset(X_train.tolist(), y_train, train_tfm)
val_ds   = VibeDataset(X_val.tolist(),   y_val,   eval_tfm)
test_ds  = VibeDataset(X_test.tolist(),  y_test,  eval_tfm)

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

In [ ]:
class CLIPLoRAClassifier(nn.Module):
    """CLIP vision tower with LoRA adapters + a linear classification head."""
    def __init__(self, clip_model, num_classes: int, lora_r=8, lora_alpha=16, lora_dropout=0.1):
        super().__init__()
        # Wrap the vision tower in LoRA. peft injects rank-r adapters at the
        # listed target modules; the rest stays frozen.
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
            lora_dropout=lora_dropout,
            bias='none',
        )
        self.vision = get_peft_model(clip_model.vision_model, lora_config)
        self.visual_projection = clip_model.visual_projection  # frozen
        for p in self.visual_projection.parameters():
            p.requires_grad = False
        self.classifier = nn.Linear(clip_model.config.projection_dim, num_classes)

    def forward(self, pixel_values):
        v = self.vision(pixel_values=pixel_values).pooler_output
        v = self.visual_projection(v)
        return self.classifier(v)

model = CLIPLoRAClassifier(clip_model, num_classes=len(CLASSES)).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.2f}M / {total/1e6:.1f}M  ({trainable/total*100:.2f}%)')

In [ ]:
@torch.no_grad()
def evaluate(loader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    all_preds, all_logits, all_y = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x).cpu().numpy()
        all_logits.append(logits)
        all_preds.append(logits.argmax(axis=1))
        all_y.append(y.numpy())
    preds = np.concatenate(all_preds)
    logits = np.concatenate(all_logits)
    y_true = np.concatenate(all_y)
    return accuracy_score(y_true, preds), preds, logits

# Different LRs for the LoRA params vs the classifier head
lora_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' in n]
optimizer = torch.optim.AdamW(
    [
        {'params': lora_params, 'lr': 1e-4, 'weight_decay': 0.01},
        {'params': head_params, 'lr': 1e-3, 'weight_decay': 0.01},
    ]
)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

EPOCHS = 30
PATIENCE = 6
best_val = 0.0
best_state = None
patience = 0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= len(train_ds)

    val_acc, _, _ = evaluate(val_loader)
    history['train_loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    print(f'epoch {epoch+1:>2}  loss={epoch_loss:.4f}  val_acc={val_acc:.3f}')

    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if v.requires_grad or True}
        patience = 0
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1} (best val_acc={best_val:.3f})')
            break

if best_state is not None:
    model.load_state_dict(best_state, strict=False)
print(f'\nLoaded best checkpoint (val_acc={best_val:.3f})')

In [ ]:
# Plot training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history['train_loss']); ax1.set_title('Train loss'); ax1.set_xlabel('epoch')
ax2.plot(history['val_acc']);    ax2.set_title('Val accuracy'); ax2.set_xlabel('epoch')
plt.tight_layout(); plt.show()

lora_acc, lora_pred, lora_logits = evaluate(test_loader)
lora_top3 = top_k_accuracy_score(
    np.concatenate([y.numpy() for _, y in test_loader]),
    lora_logits, k=3, labels=list(range(len(CLASSES))),
)
print(f'CLIP + LoRA — top-1: {lora_acc:.3f}  top-3: {lora_top3:.3f}')

## 7. Groq Vision baseline

Load the cached Groq predictions for the same test split (run
`scripts/export_groq_baseline.py` from the repo root first to populate
`data/eval/groq_predictions.json`).

If the file isn't there, this cell prints a warning and the comparison
skips Groq.

In [ ]:
groq_pred_path = DATA_ROOT / 'groq_predictions.json'
groq_pred = None
if groq_pred_path.exists():
    raw = json.loads(groq_pred_path.read_text())
    # raw is {path: predicted_class_name}
    groq_pred = np.array([
        CLASS_TO_IDX.get(raw.get(str(Path(p).resolve())), -1)
        for p in X_test
    ])
    valid = groq_pred >= 0
    groq_acc = accuracy_score(y_test[valid], groq_pred[valid])
    print(f'Groq Vision — top-1: {groq_acc:.3f}  (on {valid.sum()}/{len(y_test)} test samples)')
else:
    print(f'No Groq predictions cache at {groq_pred_path}. Run scripts/export_groq_baseline.py first.')

## 8. Comparison table

The headline result for the report.

In [ ]:
import pandas as pd

rows = [
    {'system': 'CLIP zero-shot',     'top-1': zs_acc,   'top-3': zs_top3,   'trainable_params': 0},
    {'system': 'CLIP linear probe',  'top-1': lp_acc,   'top-3': lp_top3,   'trainable_params': lp_model.coef_.size},
    {'system': 'CLIP + LoRA (ours)', 'top-1': lora_acc, 'top-3': lora_top3, 'trainable_params': trainable},
]
if groq_pred is not None:
    rows.insert(0, {'system': 'Groq Vision (Llama-4 Scout)', 'top-1': groq_acc, 'top-3': None, 'trainable_params': None})

results = pd.DataFrame(rows)
results

### 8.1 Confusion matrices

Look at which classes get confused. If `scandinavian` and `japandi`
blur together, that's a *known* hard pair and worth discussing in the
report.

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='YlOrBr')
    plt.xticks(rotation=45, ha='right')
    plt.title(title)
    plt.ylabel('true'); plt.xlabel('predicted')
    plt.tight_layout(); plt.show()

plot_cm(y_test, zs_pred,   'CLIP zero-shot')
plot_cm(y_test, lp_pred,   'CLIP linear probe')
plot_cm(y_test, lora_pred, 'CLIP + LoRA (ours)')

### 8.2 Per-class accuracy

Where does LoRA help most? Where does it hurt?

In [ ]:
print('LoRA per-class report:')
print(classification_report(y_test, lora_pred, target_names=CLASSES, zero_division=0))

## 9. Save the model

Persist the trained LoRA adapters + classification head so we can load
them in the backend later (only ~1 MB of weights — tiny). The frozen CLIP
backbone is re-downloaded from HuggingFace at inference time.

In [ ]:
SAVE_DIR = Path('artifacts/clip_lora_vibe')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save LoRA adapters
model.vision.save_pretrained(SAVE_DIR / 'lora_adapters')
# Save classification head + class list
torch.save(model.classifier.state_dict(), SAVE_DIR / 'classifier.pt')
(SAVE_DIR / 'classes.json').write_text(json.dumps(CLASSES, indent=2))
print(f'Saved to {SAVE_DIR.resolve()}')

## 10. Next steps for the report

1. **Run this notebook end-to-end** with the final photo set and screenshot the comparison table + confusion matrices for the report.
2. **Ablations** worth doing if time permits:
   - LoRA rank sweep: r=4, 8, 16, 32
   - With vs without augmentation
   - With vs without label smoothing
   - Train on 50% / 75% / 100% of data — accuracy-vs-data curve
3. **Error analysis**: pick 3–5 misclassified images, show them, write a sentence each on why the model got it wrong (e.g. "both `scandinavian` and `minimalist` rely on pale walls; the model used texture cues that flipped this one").
4. **Failure modes**: try a photo from outside the 8 trained classes (e.g. a y2k outfit). The model will confidently mislabel — that's the closed-set assumption in action and worth noting.
5. **Production decision**: if LoRA beats Groq on the test set, swap it into `vibecheck/server/app.py` as the primary classifier. Otherwise document as research and keep Groq.